In [ ]:
import sys
import os

# Use current working directory andS go one level up
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

# Now you acan import your config
from config import api_key, serper_api_key

import os

os.environ["OPENAI_MODEL_NAME"] = 'gpt-4-turbo'
os.environ["OPENAI_API_KEY"] = api_key
os.environ["SERPER_API_KEY"] = serper_api_key

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from crewai import Agent, Task, Crew

In [ ]:
from crewai_tools import SerperDevTool

serper_tool = SerperDevTool(
    search_url="https://google.serper.dev/scholar",
    n_results=10,
    country='nl'
)

print(serper_tool.run(search_query="ChatGPT"))


In [ ]:
from crewai_tools import ScrapeWebsiteTool

# Initialize tool (optional URL can be passed later)
scrape_tool = ScrapeWebsiteTool()

# Use the tool to fetch content
text = scrape_tool.run(website_url="https://example.com")
print(text)

In [ ]:
# Agent 1: Researcher
researcher = Agent(
    role="soccer sports analyst",
    goal="find the best sources on the internet that have historical data on dutch 'eredivisie' matches.",
    tools = [serper_tool],
    verbose=True,
    backstory=(
        "As a soccer sports statistician, you navigate the internet in "
        "search of up to date and accurate historical data for soccer "
        "matches and historical players data."
        "The data must be of high quality and must range from today up to 2 years in the past."
        "The data must be from the Dutch 'eredivisie."
    )
)

In [ ]:
# Agent 1: Researcher
data_collector = Agent(
    role="soccer sports data collector",
    goal="to gather historical data of dutch 'eredivisie' matches "
         "and historical data of player statistics ",
    tools = [scrape_tool],
    verbose=True,
    backstory=(
        "As a soccer sports data collector you use the result of the soccer sport analyst. "
        "You take the number one rated source from the soccer sports analyst and scrape all matches from that source."
    )
)

In [ ]:
# Task for Researcher Agent: Extract Job Requirements
research_task = Task(
    description=(
        "1. Find 3 usefull sources for historical data of Dutch eredivisie matches."
        #"2. Do not allow https://eredivisie.eu/ to be in your final results as the site is not complete. "
        "3. The 3 sources must have data on historical matches and statics of each match. The data of the"
        " season {season} must be complete and must contain additional data like shots at goals, corners, penalty, faults, etc"
        " if it is not complete another sources must be searched for."
        "4. Score each resource based on the following criteria free of costs, availability, completeness, accuracy. "
        "5. Order the sources in based on the provided criteria. The best on first."
    ),
    expected_output=(
        "The output must be a table that can be rendered as markdown in a jupyter notebook."
        "The must include a clickable url that leads to the historical matches."
    ),
    agent=researcher,
    output_file='best_sources.md',
    async_execution=False
)

In [ ]:
# Task for Profiler Agent: Compile Comprehensive Profile
data_task = Task(
    description=(
        "1. scrape all matches from season {season} for the dutch eredivisie from the best ranked source provided by the soccer sport analyst. "
        "2. data might be available under scores and fixture tab (e.g.https://fbref.com/en/comps/23/2022-2023/schedule/2022-2023-Eredivisie-Scores-and-Fixtures) and easy to scrape. "
        "3. if the data cannot be scraped for the sources ranked as number one continue with number 2 and then number 3. Stop as soon as one has succeeded. " 
    ),
    expected_output=(
        "The output should be in csv formal where the first row contains the headers of the columns."
    ),
    agent=data_collector,
    output_file='data.csv',
    async_execution=False
)

In [ ]:
job_application_crew = Crew(
    agents=[researcher, data_collector ],
    tasks=[research_task, data_task],
    verbose=True
)

In [ ]:
inputs = {"season": "2023/2024"}

In [ ]:
### this execution will take a few minutes to run
result = job_application_crew.kickoff(inputs=inputs)

In [ ]:
from IPython.display import Markdown
Markdown(result.raw)

In [ ]:
from IPython.display import Markdown
Markdown("best_sources.md")

# Part 2

In [ ]:
from crewai_tools import SerperDevTool

serper_tool = SerperDevTool()

In [ ]:
# Agent 1: Researcher
dutch_links_finder = Agent(
    role="Internet searcher for Dutch Eredivisie football",
    goal="Find all team pages on fbref.com for the Eredivisie {season}, one per team (18 in total).",
    tools=[serper_tool],
    verbose=True,
    output_file='urls.json',
    backstory=(
        "We need an exact list of 18 teams from the Dutch Eredivisie season {season}, "
        "where each team entry contains its name and its matches URL from FBref."
    )
)

# Task for Researcher Agent
dutch_link_task = Task(
    description=(
        "1. Go to https://fbref.com/ and navigate to the Eredivisie season {season}. "
        "2. Extract all 18 team pages. "
        "3. Each page looks like this example for English football: "
        "'https://fbref.com/en/squads/b8fd03ef/2022-2023/Manchester-City-Stats'. "
        "4. Create a JSON list of 18 objects, each object containing two keys: "
        "'team' (string, team name) and 'url' (string, the fbref URL). "
        "5. Validate that there are 18 different elemenst each related to 1 club. If not you have to reiterate until you have 18."
    ),
    expected_output=(
        "A valid JSON array of length 18. Example:\n"
        '[{"team": "Ajax", "url": "https://..."}, {"team": "PSV", "url": "https://..."}, ...]\n'
        "Do not include any text, comments, or markdown formatting — only valid JSON."
    ),
    agent=dutch_links_finder,
    output_file="urls.json",
    async_execution=False
)


In [ ]:
job_application_crew = Crew(
    agents=[dutch_links_finder ],
    tasks=[dutch_link_task],
    verbose=True
)

In [ ]:
### this execution will take a few minutes to run
result = job_application_crew.kickoff(inputs={"season":"2023-2024"})

In [ ]:
import json

with open("urls.json","r") as f:
    data = json.load(f)

for club in data:
    print(club)

# Part 3

In [ ]:
from crewai import Agent, Task, Crew
from crewai.tools import BaseTool

# -------------------------------
# Tool: Research Assistant
# -------------------------------
class FootballResearchTool(BaseTool):
    name: str = "FootballResearchTool"
    description: str = "Researches best features and models to predict football match outcomes."

    def _run(self, query: str) -> str:
        # This is a simple placeholder: in reality, you could integrate APIs or search engines
        output = (
            "Top features: team form, head-to-head, goals scored, injuries, home advantage\n"
            "Top models: Logistic Regression, Random Forest, XGBoost, LightGBM\n"
        )
        return output

from crewai_tools import SerperDevTool

serper_tool = SerperDevTool()

research_tool = FootballResearchTool()

# -------------------------------
# Agent
# -------------------------------
research_agent = Agent(
    role="Researcher to find best prediction features",
    goal="to find the best features that from historical data that best predict the outcome of a football match.",
    tools=[serper_tool],
    backstory=(
    "I want to estimate the ourcome of football matches for this I need the most significant prediction factors called feature. "
)

)

# -------------------------------
# Task
# -------------------------------
research_task = Task(
    description="Research best features and models for predicting football match outcomes. "
                "The best should be based on evidence or published articles."
                "The features must have a single purpose and not be vague. So historical match results is vague. What does this mean? "
                "how can each feature be constructed from match data, player data or other data "
                "try to be as concrete as possible, also how it can be constructed from for example historical match data,"
                "sources can be kaggle, medium, articles, publications, etc. Be creative",
    expected_output="A list of the 10 best features that can predict soccer match outcomes."
                    "For each features provide a clear definition, also the reasoning why it a good features. ",
    agent=research_agent,
    async_execution=False
)

# -------------------------------
# Crew
# -------------------------------
football_crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    verbose=True
)

# -------------------------------
# Kick it off
# -------------------------------
result = football_crew.kickoff()


In [ ]:
from IPython.display import Markdown
Markdown(result.raw)